# `defaultdict` — Advanced Problems with Solutions (Tutorial Style)

This notebook is a second, independent set of advanced exercises on `collections.defaultdict`.

The approach here is deliberately tutorial-oriented:

- we start with a concrete problem;
- we often solve it first with a regular dictionary;
- we identify the repetitive missing-key logic;
- we replace that logic with an appropriate default factory;
- we test edge cases;
- and then we discuss what changed and why.

The goal is not merely to memorize `defaultdict(int)` or `defaultdict(list)`, but to learn how to recognize the *shape of a missing value* in real programs.


Before we begin, let us import the tools used throughout the notebook.

Everything here uses only the Python standard library.


In [1]:
from collections import defaultdict, deque
from functools import partial, wraps
from itertools import combinations
from pprint import pprint
from datetime import datetime, timezone


A useful mental model is this:

> A `defaultdict` answers the question: **“If this key does not exist yet, what fresh value should be created for it?”**

Typical answers are:

- `0` → use `int`
- `0.0` → use `float`
- `[]` → use `list`
- `set()` → use `set`
- `{}` → use `dict`
- a richer record → use a custom factory function

We will keep returning to that question.


## Problem 1 — Build an anagram index

Suppose we receive a collection of words and want to group together words that are anagrams of each other.

For example, `listen`, `silent`, and `enlist` should end up in the same group.

A convenient key is the sorted sequence of letters in each word.


Let us start by defining the data.


In [2]:
words = [
    'listen', 'silent', 'enlist',
    'evil', 'vile', 'veil', 'live',
    'stone', 'tones', 'notes',
    'python'
]


### Step 1 — Solve it with a regular dictionary

With a normal dictionary, every time we see a signature for the first time we need to create a list.


In [3]:
groups = {}

for word in words:
    signature = ''.join(sorted(word))
    if signature in groups:
        groups[signature].append(word)
    else:
        groups[signature] = [word]

pprint(groups)


{'eilnst': ['listen', 'silent', 'enlist'],
 'eilv': ['evil', 'vile', 'veil', 'live'],
 'enost': ['stone', 'tones', 'notes'],
 'hnopty': ['python']}


This works, but the `if/else` is not really about the problem itself.

It exists only because missing dictionary keys do not automatically contain an empty list.


### Step 2 — Replace the missing-key branch with `defaultdict(list)`

The natural missing value is a **fresh empty list**.


In [4]:
anagram_index = defaultdict(list)

for word in words:
    signature = ''.join(sorted(word))
    anagram_index[signature].append(word)

pprint(dict(anagram_index))


{'eilnst': ['listen', 'silent', 'enlist'],
 'eilv': ['evil', 'vile', 'veil', 'live'],
 'enost': ['stone', 'tones', 'notes'],
 'hnopty': ['python']}


Now the loop describes the real operation directly: compute a signature, then append the word to that group.


### Step 3 — Package the solution as a function


In [5]:
def build_anagram_index(words):
    index = defaultdict(list)

    for word in words:
        signature = ''.join(sorted(word.lower()))
        index[signature].append(word)

    return dict(index)

result = build_anagram_index(words)

assert sorted(result['eilnst']) == ['enlist', 'listen', 'silent']
assert sorted(result['eilv']) == ['evil', 'live', 'veil', 'vile']
assert result['hnopty'] == ['python']


The important design choice was not “use `defaultdict`.”

It was first recognizing that the missing state for each signature is an **independent empty list**.


## Problem 2 — Count state transitions

Imagine we are analyzing user navigation through application states.

For each observed transition `(from_state, to_state)`, we want to count how many times it occurred.

We will store the result as:

```text
from_state -> to_state -> count
```

This requires two levels of missing-key behavior.


In [6]:
transitions = [
    ('home', 'search'),
    ('search', 'product'),
    ('product', 'cart'),
    ('home', 'search'),
    ('search', 'home'),
    ('search', 'product'),
    ('product', 'home'),
]


### Step 1 — Think about the inner value

For a fixed source state such as `search`, we want another dictionary whose missing destination count begins at zero.

That inner structure is therefore:

```python
defaultdict(int)
```


### Step 2 — Make the outer mapping create inner mappings

The outer factory can be a lambda that creates a new `defaultdict(int)`.


In [7]:
transition_counts = defaultdict(lambda: defaultdict(int))

for source, target in transitions:
    transition_counts[source][target] += 1

pprint({
    source: dict(targets)
    for source, targets in transition_counts.items()
})


{'home': {'search': 2},
 'product': {'cart': 1, 'home': 1},
 'search': {'home': 1, 'product': 2}}


Notice the two automatic creations that can happen in one line:

```python
transition_counts[source][target] += 1
```

If `source` is missing, an inner dictionary is created.

If `target` is then missing inside that dictionary, its count starts at zero.


### Step 3 — Write a reusable function and verify the result


In [8]:
def count_transitions(records):
    counts = defaultdict(lambda: defaultdict(int))

    for source, target in records:
        counts[source][target] += 1

    return {
        source: dict(targets)
        for source, targets in counts.items()
    }

counts = count_transitions(transitions)

assert counts['home']['search'] == 2
assert counts['search']['product'] == 2
assert counts['search']['home'] == 1
assert counts['product']['cart'] == 1


This pattern appears in transition matrices, clickstream analysis, Markov-model preprocessing, workflow statistics, and graph edge counting.


## Problem 3 — Build a bidirectional lookup table

Suppose we have pairs `(category, item)`.

We want both of these mappings:

- category → items
- item → categories

Duplicates should not matter, so a set is more appropriate than a list.


In [9]:
pairs = [
    ('backend', 'python'),
    ('backend', 'redis'),
    ('data', 'python'),
    ('data', 'sql'),
    ('backend', 'python'),  # duplicate
    ('analytics', 'sql'),
]


### Step 1 — Choose the correct factory

A category may contain each item at most once, so the missing value should be a fresh empty set.

That suggests `defaultdict(set)`.


In [10]:
category_to_items = defaultdict(set)
item_to_categories = defaultdict(set)

for category, item in pairs:
    category_to_items[category].add(item)
    item_to_categories[item].add(category)

pprint(dict(category_to_items))
pprint(dict(item_to_categories))


{'analytics': {'sql'},
 'backend': {'redis', 'python'},
 'data': {'sql', 'python'}}
{'python': {'data', 'backend'},
 'redis': {'backend'},
 'sql': {'data', 'analytics'}}


### Step 2 — Why not `list`?

If we used a list, the repeated pair `('backend', 'python')` would appear twice unless we wrote extra deduplication logic.

The factory should reflect the invariant we actually want.


In [11]:
assert category_to_items['backend'] == {'python', 'redis'}
assert item_to_categories['python'] == {'backend', 'data'}
assert item_to_categories['sql'] == {'data', 'analytics'}


## Problem 4 — Group events into time buckets

We receive events as `(minute, event_type, payload)`.

We want:

```text
minute -> event_type -> list of payloads
```

This is a nested grouping problem rather than a nested counting problem.


In [12]:
events = [
    (10, 'click', {'id': 1}),
    (10, 'view', {'id': 2}),
    (10, 'click', {'id': 3}),
    (11, 'view', {'id': 4}),
    (12, 'purchase', {'id': 5}),
    (12, 'view', {'id': 6}),
]


### Step 1 — Determine the leaf value

At the deepest level, each `(minute, event_type)` pair needs a list.

So the inner mapping should be `defaultdict(list)`.


### Step 2 — Make each new minute create that inner mapping


In [13]:
buckets = defaultdict(lambda: defaultdict(list))

for minute, event_type, payload in events:
    buckets[minute][event_type].append(payload)

pprint({
    minute: {kind: values for kind, values in kinds.items()}
    for minute, kinds in buckets.items()
})


{10: {'click': [{'id': 1}, {'id': 3}], 'view': [{'id': 2}]},
 11: {'view': [{'id': 4}]},
 12: {'purchase': [{'id': 5}], 'view': [{'id': 6}]}}


### Step 3 — Query the result


In [14]:
assert [x['id'] for x in buckets[10]['click']] == [1, 3]
assert [x['id'] for x in buckets[12]['view']] == [6]


The nested factory is doing structural work for us, but the data model remains explicit:

- outer key: minute
- inner key: event type
- leaf value: list of payloads


## Problem 5 — Per-customer running statistics

For every customer we want to maintain:

- number of purchases;
- total amount spent;
- largest single purchase.

A simple built-in factory such as `int` or `list` is no longer enough.


In [15]:
purchases = [
    ('alice', 19.99),
    ('bob', 50.00),
    ('alice', 120.00),
    ('alice', 4.50),
    ('bob', 10.00),
    ('carol', 75.25),
]


### Step 1 — Define the missing record

Whenever a customer is first seen, we want a fresh statistics dictionary.


In [16]:
def new_customer_stats():
    return {
        'count': 0,
        'total': 0.0,
        'largest': 0.0,
    }


### Step 2 — Use the custom function as the default factory


In [17]:
stats = defaultdict(new_customer_stats)

for customer, amount in purchases:
    record = stats[customer]
    record['count'] += 1
    record['total'] += amount
    record['largest'] = max(record['largest'], amount)

pprint(dict(stats))


{'alice': {'count': 3, 'largest': 120.0, 'total': 144.49},
 'bob': {'count': 2, 'largest': 50.0, 'total': 60.0},
 'carol': {'count': 1, 'largest': 75.25, 'total': 75.25}}


The factory is called only when a customer is accessed for the first time.

After that, the same record is updated repeatedly.


In [18]:
assert stats['alice']['count'] == 3
assert abs(stats['alice']['total'] - 144.49) < 1e-9
assert stats['alice']['largest'] == 120.00
assert stats['carol']['count'] == 1


## Problem 6 — Detect a shared-mutable factory bug

This problem is about a subtle mistake.

We want every missing key to receive its own list.

Consider the following factory.


In [19]:
shared_list = []
bad = defaultdict(lambda: shared_list)


At first glance, this looks like a function that returns a list.

But it returns the *same* list every time.


In [20]:
bad['a'].append(1)
bad['b'].append(2)

print('a:', bad['a'])
print('b:', bad['b'])
print('same object:', bad['a'] is bad['b'])


a: [1, 2]
b: [1, 2]
same object: True


Both keys point to one shared mutable object.

That is almost never what we want for grouping.


### Step 2 — Fix the factory

`list` itself is a callable. Calling `list()` creates a new list.


In [21]:
good = defaultdict(list)

good['a'].append(1)
good['b'].append(2)

assert good['a'] == [1]
assert good['b'] == [2]
assert good['a'] is not good['b']


A good rule is:

> If the default value is mutable, the factory should normally create a **new object on every call**.


## Problem 7 — Read-only lookup without accidental insertion

A `defaultdict` can mutate when it is merely indexed.

That is useful during construction, but dangerous in reporting code.


In [22]:
stock = defaultdict(int, apples=8, pears=3)

print(dict(stock))


{'apples': 8, 'pears': 3}


### Step 1 — See the side effect

We ask for a product that does not exist.


In [23]:
quantity = stock['bananas']

print(quantity)
print(dict(stock))


0
{'apples': 8, 'pears': 3, 'bananas': 0}


The lookup returned zero, but it also inserted `'bananas': 0`.

Sometimes that is exactly what we want.

Sometimes it is a bug.


### Step 2 — Use `.get()` for a non-mutating read


In [24]:
stock2 = defaultdict(int, apples=8, pears=3)

before = dict(stock2)
quantity = stock2.get('bananas', 0)
after = dict(stock2)

assert quantity == 0
assert before == after
assert 'bananas' not in stock2


The distinction is worth remembering:

```python
d[key]      # may invoke the factory and insert the key

d.get(key)  # does not invoke the defaultdict factory
```


## Problem 8 — Build an inverted search index with positions

Instead of mapping a word merely to documents, we will map each word to the positions at which it appears inside each document.

The structure will be:

```text
word -> document_id -> list of positions
```

This requires nested list creation.


In [25]:
documents = {
    1: 'red blue red green',
    2: 'blue green blue',
    3: 'green red',
}


### Step 1 — Decide what a missing document entry should contain

For a fixed word and document, we need an empty list of positions.

So the inner mapping is `defaultdict(list)`.


### Step 2 — Make a missing word create that inner mapping


In [26]:
position_index = defaultdict(lambda: defaultdict(list))

for doc_id, text in documents.items():
    for position, word in enumerate(text.split()):
        position_index[word][doc_id].append(position)

pprint({
    word: dict(doc_map)
    for word, doc_map in position_index.items()
})


{'blue': {1: [1], 2: [0, 2]},
 'green': {1: [3], 2: [1], 3: [0]},
 'red': {1: [0, 2], 3: [1]}}


### Step 3 — Verify exact positions


In [27]:
assert position_index['red'][1] == [0, 2]
assert position_index['blue'][2] == [0, 2]
assert position_index['green'][3] == [0]


This is a useful pattern for text indexing, token offsets, event positions, and sequence analysis.


## Problem 9 — Group overlapping intervals by owner

We receive intervals `(owner, start, end)`.

First group intervals by owner, then merge any overlapping intervals for each owner.

`defaultdict` will solve the grouping step; the merging step is regular algorithmic logic.


In [28]:
intervals = [
    ('alice', 1, 4),
    ('bob', 2, 3),
    ('alice', 3, 7),
    ('alice', 10, 12),
    ('bob', 8, 10),
    ('alice', 11, 15),
]


### Step 1 — Group intervals


In [29]:
by_owner = defaultdict(list)

for owner, start, end in intervals:
    by_owner[owner].append((start, end))

pprint(dict(by_owner))


{'alice': [(1, 4), (3, 7), (10, 12), (11, 15)], 'bob': [(2, 3), (8, 10)]}


### Step 2 — Write a normal interval-merging function

The important point is that `defaultdict` should simplify the missing-key problem, not replace unrelated algorithmic work.


In [30]:
def merge_intervals(items):
    items = sorted(items)
    if not items:
        return []

    merged = [list(items[0])]

    for start, end in items[1:]:
        last_start, last_end = merged[-1]

        if start <= last_end:
            merged[-1][1] = max(last_end, end)
        else:
            merged.append([start, end])

    return [tuple(item) for item in merged]


### Step 3 — Apply it to every group


In [31]:
merged_by_owner = {
    owner: merge_intervals(owner_intervals)
    for owner, owner_intervals in by_owner.items()
}

pprint(merged_by_owner)

assert merged_by_owner['alice'] == [(1, 7), (10, 15)]
assert merged_by_owner['bob'] == [(2, 3), (8, 10)]


{'alice': [(1, 7), (10, 15)], 'bob': [(2, 3), (8, 10)]}


## Problem 10 — Build a weighted undirected graph

Each record `(u, v, weight)` adds weight to an undirected edge.

Repeated edges should accumulate.

We want:

```text
node -> neighbor -> total_weight
```


In [32]:
weighted_edges = [
    ('A', 'B', 2.5),
    ('A', 'C', 1.0),
    ('B', 'A', 0.5),
    ('B', 'C', 3.0),
    ('A', 'B', 1.0),
]


### Step 1 — Use nested float accumulation

A missing neighbor weight should start at `0.0`.


In [33]:
graph = defaultdict(lambda: defaultdict(float))

for u, v, weight in weighted_edges:
    graph[u][v] += weight
    graph[v][u] += weight


Because the graph is undirected, every input contributes in both directions.


In [34]:
pprint({node: dict(neighbors) for node, neighbors in graph.items()})

assert graph['A']['B'] == 4.0
assert graph['B']['A'] == 4.0
assert graph['A']['C'] == 1.0
assert graph['C']['B'] == 3.0


{'A': {'B': 4.0, 'C': 1.0},
 'B': {'A': 4.0, 'C': 3.0},
 'C': {'A': 1.0, 'B': 3.0}}


The nested dictionary is doing two jobs:

- discovering nodes as they appear;
- discovering neighbors as they appear.

The `float` factory handles the numeric identity value for accumulation.


## Problem 11 — Generate a contingency table

Suppose we have survey responses `(department, answer)`.

We want a two-dimensional frequency table:

```text
department -> answer -> count
```

Then we want row totals and column totals.


In [35]:
responses = [
    ('Engineering', 'Yes'),
    ('Engineering', 'No'),
    ('Engineering', 'Yes'),
    ('Sales', 'Yes'),
    ('Sales', 'Maybe'),
    ('Sales', 'Yes'),
    ('HR', 'No'),
]


### Step 1 — Build the table


In [36]:
table = defaultdict(lambda: defaultdict(int))

for department, answer in responses:
    table[department][answer] += 1

pprint({department: dict(row) for department, row in table.items()})


{'Engineering': {'No': 1, 'Yes': 2},
 'HR': {'No': 1},
 'Sales': {'Maybe': 1, 'Yes': 2}}


### Step 2 — Compute row totals


In [37]:
row_totals = {
    department: sum(row.values())
    for department, row in table.items()
}

print(row_totals)


{'Engineering': 3, 'Sales': 3, 'HR': 1}


### Step 3 — Compute column totals

A second `defaultdict(int)` is a convenient accumulator.


In [38]:
column_totals = defaultdict(int)

for row in table.values():
    for answer, count in row.items():
        column_totals[answer] += count

print(dict(column_totals))

assert row_totals['Engineering'] == 3
assert column_totals['Yes'] == 4
assert column_totals['No'] == 2
assert column_totals['Maybe'] == 1


{'Yes': 4, 'No': 2, 'Maybe': 1}


## Problem 12 — Use a stateful factory

A default factory does not have to return a constant value.

It can maintain its own state.

We will create unique numeric IDs for previously unseen labels.


### Step 1 — Create a factory with a counter


In [39]:
def make_id_factory(start=1000):
    next_id = start

    def factory():
        nonlocal next_id
        value = next_id
        next_id += 1
        return value

    return factory


Every call to the returned factory produces the next ID.


### Step 2 — Use that factory in a `defaultdict`


In [40]:
label_to_id = defaultdict(make_id_factory(1000))

print(label_to_id['alpha'])
print(label_to_id['beta'])
print(label_to_id['alpha'])
print(label_to_id['gamma'])


1000
1001
1000
1002


The same key does not call the factory twice.

Only the first access to a missing key allocates a new ID.


In [41]:
assert label_to_id['alpha'] == 1000
assert label_to_id['beta'] == 1001
assert label_to_id['gamma'] == 1002
assert len(label_to_id) == 3


Stateful factories can be powerful, but they also make behavior less obvious.

Use them when the stateful creation rule is genuinely part of the model.


## Problem 13 — Create a reusable record constructor with `partial`

Suppose many record dictionaries should return `'unknown'` for absent fields.

Writing the same lambda every time is repetitive.


### Step 1 — Create a specialized constructor

We can partially apply the first argument of `defaultdict`.


In [42]:
unknown_record = partial(defaultdict, lambda: 'unknown')


Now `unknown_record(...)` behaves like a small custom constructor.


In [43]:
alice = unknown_record(name='Alice', role='Engineer')
bob = unknown_record(name='Bob')

assert alice['role'] == 'Engineer'
assert bob['role'] == 'unknown'
assert bob['country'] == 'unknown'

print(dict(alice))
print(dict(bob))


{'name': 'Alice', 'role': 'Engineer'}
{'name': 'Bob', 'role': 'unknown', 'country': 'unknown'}


Remember that indexing a missing field inserts it.

After `bob['country']`, the key `'country'` really exists in `bob`.


## Problem 14 — Track first and last observations

For each sensor, record:

- how many readings were seen;
- the first timestamp;
- the last timestamp;
- the running sum.

This requires a custom record whose initial values depend on the first reading.

That creates an interesting limitation: the default factory receives **no key and no reading value**.


In [44]:
readings = [
    ('s1', 100, 10.0),
    ('s2', 101, 20.0),
    ('s1', 105, 12.0),
    ('s1', 110, 11.0),
    ('s2', 120, 18.0),
]


### Step 1 — Use a neutral record

Because the factory cannot see the incoming timestamp, we initialize timestamp fields with `None`.


In [45]:
def new_sensor_record():
    return {
        'count': 0,
        'first_timestamp': None,
        'last_timestamp': None,
        'sum': 0.0,
    }


### Step 2 — Fill in the first timestamp during the update


In [46]:
sensor_stats = defaultdict(new_sensor_record)

for sensor, timestamp, value in readings:
    record = sensor_stats[sensor]

    if record['count'] == 0:
        record['first_timestamp'] = timestamp

    record['count'] += 1
    record['last_timestamp'] = timestamp
    record['sum'] += value

pprint(dict(sensor_stats))


{'s1': {'count': 3, 'first_timestamp': 100, 'last_timestamp': 110, 'sum': 33.0},
 's2': {'count': 2, 'first_timestamp': 101, 'last_timestamp': 120, 'sum': 38.0}}


The lesson here is important:

> A `defaultdict` factory is a no-argument constructor. If initialization needs information from the missing key or the current input record, some initialization may still belong in the update logic.


In [47]:
assert sensor_stats['s1']['first_timestamp'] == 100
assert sensor_stats['s1']['last_timestamp'] == 110
assert sensor_stats['s1']['count'] == 3
assert sensor_stats['s2']['sum'] == 38.0


## Problem 15 — Compute a collaborative co-occurrence graph

Each session contains a set of items viewed together.

For every pair of distinct items in the same session, increase their co-occurrence count.

We want a symmetric nested structure:

```text
item -> other_item -> count
```


In [48]:
sessions = [
    ['A', 'B', 'C'],
    ['A', 'B'],
    ['B', 'C', 'D'],
    ['A', 'D'],
]


### Step 1 — Generate unique pairs within each session

We use `combinations` so that a pair is counted once per session.


In [49]:
cooccurrence = defaultdict(lambda: defaultdict(int))

for session in sessions:
    unique_items = sorted(set(session))

    for left, right in combinations(unique_items, 2):
        cooccurrence[left][right] += 1
        cooccurrence[right][left] += 1


### Step 2 — Inspect the graph


In [50]:
pprint({item: dict(neighbors) for item, neighbors in cooccurrence.items()})

assert cooccurrence['A']['B'] == 2
assert cooccurrence['B']['C'] == 2
assert cooccurrence['B']['D'] == 1
assert cooccurrence['D']['A'] == 1


{'A': {'B': 2, 'C': 1, 'D': 1},
 'B': {'A': 2, 'C': 2, 'D': 1},
 'C': {'A': 1, 'B': 2, 'D': 1},
 'D': {'A': 1, 'B': 1, 'C': 1}}


This same pattern is useful in recommendation systems, basket analysis, social graphs, and feature co-occurrence studies.


## Problem 16 — Breadth-first search with grouped distances

We will combine a regular graph traversal with `defaultdict(list)`.

The graph traversal computes the shortest distance from a starting node.

Then we group nodes by distance:

```text
distance -> list of nodes
```


In [51]:
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': [],
}


### Step 1 — Compute shortest distances with BFS


In [52]:
def bfs_distances(graph, start):
    distances = {start: 0}
    queue = deque([start])

    while queue:
        node = queue.popleft()

        for neighbor in graph.get(node, []):
            if neighbor not in distances:
                distances[neighbor] = distances[node] + 1
                queue.append(neighbor)

    return distances

distances = bfs_distances(graph, 'A')


### Step 2 — Group nodes by distance

Now the natural missing value is an empty list.


In [53]:
levels = defaultdict(list)

for node, distance in distances.items():
    levels[distance].append(node)

for distance in levels:
    levels[distance].sort()

pprint(dict(levels))

assert levels[0] == ['A']
assert levels[1] == ['B', 'C']
assert levels[2] == ['D', 'E', 'F']


{0: ['A'], 1: ['B', 'C'], 2: ['D', 'E', 'F']}


This illustrates a broader principle: `defaultdict` is often one component inside a larger algorithm, not the entire algorithm.


## Problem 17 — Build a multi-level inventory ledger

Transactions have the form:

```text
(warehouse, sku, movement_type, quantity)
```

We want to accumulate quantities separately for `in` and `out` movements:

```text
warehouse -> sku -> movement_type -> total_quantity
```

This is a three-level structure.


In [54]:
movements = [
    ('W1', 'SKU1', 'in', 10),
    ('W1', 'SKU1', 'out', 3),
    ('W1', 'SKU2', 'in', 5),
    ('W2', 'SKU1', 'in', 7),
    ('W1', 'SKU1', 'in', 4),
    ('W2', 'SKU1', 'out', 2),
]


### Step 1 — Build the factory from the inside out

At the deepest level we need integer totals.

So:

```python
movement_type -> int
```

is `defaultdict(int)`.

Then each SKU needs one of those dictionaries.

Then each warehouse needs a dictionary of SKUs.


In [55]:
ledger = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(int)
    )
)

for warehouse, sku, movement_type, quantity in movements:
    ledger[warehouse][sku][movement_type] += quantity


### Step 2 — Derive current stock

Current stock is total `in` minus total `out`.


In [56]:
current_stock = defaultdict(dict)

for warehouse, sku_map in ledger.items():
    for sku, movement_map in sku_map.items():
        current_stock[warehouse][sku] = (
            movement_map['in'] - movement_map['out']
        )

pprint(dict(current_stock))

assert current_stock['W1']['SKU1'] == 11
assert current_stock['W1']['SKU2'] == 5
assert current_stock['W2']['SKU1'] == 5


{'W1': {'SKU1': 11, 'SKU2': 5}, 'W2': {'SKU1': 5}}


Deeply nested `defaultdict` structures are powerful, but readability matters.

If the nesting becomes difficult to understand, a named factory function is often clearer than multiple lambdas.


## Problem 18 — Capstone: request analytics by service and endpoint

We will finish with a larger problem that combines several `defaultdict` patterns.

Each request contains:

```text
(service, endpoint, status_code, latency_ms, user_id)
```

We want, in one pass:

1. request count per service;
2. unique users per service;
3. status-code counts per service and endpoint;
4. total latency per service and endpoint;
5. request count per service and endpoint;
6. average latency per service and endpoint.


In [57]:
requests = [
    ('auth', '/login', 200, 120.0, 'u1'),
    ('auth', '/login', 401, 80.0, 'u2'),
    ('auth', '/login', 200, 100.0, 'u1'),
    ('auth', '/logout', 204, 40.0, 'u1'),
    ('billing', '/invoice', 200, 250.0, 'u3'),
    ('billing', '/invoice', 500, 400.0, 'u4'),
    ('billing', '/invoice', 200, 200.0, 'u3'),
]


### Step 1 — Create one accumulator per metric

Different metrics have different natural missing values.

Counts use `int`.

Unique users use `set`.

Nested endpoint metrics use nested dictionaries.


In [58]:
request_count_by_service = defaultdict(int)
users_by_service = defaultdict(set)
status_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
latency_totals = defaultdict(lambda: defaultdict(float))
endpoint_counts = defaultdict(lambda: defaultdict(int))


### Step 2 — Process each request exactly once


In [59]:
for service, endpoint, status, latency, user_id in requests:
    request_count_by_service[service] += 1
    users_by_service[service].add(user_id)
    status_counts[service][endpoint][status] += 1
    latency_totals[service][endpoint] += latency
    endpoint_counts[service][endpoint] += 1


### Step 3 — Derive averages from totals and counts

We do not need to store every latency value.


In [60]:
average_latency = defaultdict(dict)

for service, endpoint_map in latency_totals.items():
    for endpoint, total_latency in endpoint_map.items():
        average_latency[service][endpoint] = (
            total_latency / endpoint_counts[service][endpoint]
        )


### Step 4 — Inspect a plain-dictionary view


In [61]:
report = {
    'request_count_by_service': dict(request_count_by_service),
    'users_by_service': {
        service: set(users)
        for service, users in users_by_service.items()
    },
    'status_counts': {
        service: {
            endpoint: dict(counts)
            for endpoint, counts in endpoint_map.items()
        }
        for service, endpoint_map in status_counts.items()
    },
    'average_latency': {
        service: dict(endpoint_map)
        for service, endpoint_map in average_latency.items()
    },
}

pprint(report)


{'average_latency': {'auth': {'/login': 100.0, '/logout': 40.0},
                     'billing': {'/invoice': 283.3333333333333}},
 'request_count_by_service': {'auth': 4, 'billing': 3},
 'status_counts': {'auth': {'/login': {200: 2, 401: 1}, '/logout': {204: 1}},
                   'billing': {'/invoice': {200: 2, 500: 1}}},
 'users_by_service': {'auth': {'u1', 'u2'}, 'billing': {'u4', 'u3'}}}


### Step 5 — Verify important invariants


In [62]:
assert report['request_count_by_service'] == {
    'auth': 4,
    'billing': 3,
}

assert report['users_by_service']['auth'] == {'u1', 'u2'}
assert report['users_by_service']['billing'] == {'u3', 'u4'}

assert report['status_counts']['auth']['/login'][200] == 2
assert report['status_counts']['auth']['/login'][401] == 1
assert report['status_counts']['billing']['/invoice'][500] == 1

assert abs(report['average_latency']['auth']['/login'] - 100.0) < 1e-12
assert abs(report['average_latency']['billing']['/invoice'] - (850.0 / 3)) < 1e-12


This capstone shows why selecting the correct factory matters.

We used:

- `int` for counts;
- `set` for unique membership;
- `float` for numeric accumulation;
- nested factories where dimensions were hierarchical.

The code remains compact because each dictionary knows how to initialize its own missing values.


# More short advanced exercises

The next exercises are smaller, but each highlights a subtle behavior worth understanding.


## Mini-problem A — Does membership testing call the factory?

Let us instrument a factory so that we can count how often it runs.


In [63]:
factory_calls = 0

def tracked_factory():
    global factory_calls
    factory_calls += 1
    return []

tracked = defaultdict(tracked_factory)

assert ('x' in tracked) is False
assert factory_calls == 0

tracked['x']
assert factory_calls == 1

tracked['x']
assert factory_calls == 1


Membership testing with `in` does not invoke the default factory.

Missing-key indexing does.


## Mini-problem B — What does `setdefault` do on a `defaultdict`?

`setdefault` follows normal dictionary semantics and uses the default value passed to `setdefault` itself.


In [64]:
d = defaultdict(list)

value = d.setdefault('a', ['manual'])

assert value == ['manual']
assert d['a'] == ['manual']


The `list` factory was not needed because `setdefault` inserted the explicit list we supplied.


## Mini-problem C — Freeze a `defaultdict` after construction

A useful pattern is to build permissively and then make future missing keys fail.


In [65]:
counts = defaultdict(int)

for ch in 'mississippi':
    counts[ch] += 1

counts.default_factory = None

assert counts['i'] == 4

try:
    counts['z']
except KeyError:
    print('Missing keys are now strict.')
else:
    raise AssertionError('Expected KeyError')


Missing keys are now strict.


This is useful when the construction phase is over and accidental new keys should be treated as errors.


## Mini-problem D — Convert nested `defaultdict` values recursively

A shallow `dict(d)` conversion only converts the outer mapping.

Nested `defaultdict` objects remain nested `defaultdict` objects.


In [66]:
nested = defaultdict(lambda: defaultdict(int))
nested['A']['x'] = 1

shallow = dict(nested)
assert isinstance(shallow['A'], defaultdict)


So let us write a recursive conversion helper.


In [67]:
def to_plain(value):
    if isinstance(value, defaultdict):
        return {
            key: to_plain(child)
            for key, child in value.items()
        }

    if isinstance(value, dict):
        return {
            key: to_plain(child)
            for key, child in value.items()
        }

    return value

plain = to_plain(nested)

assert plain == {'A': {'x': 1}}
assert isinstance(plain, dict)
assert isinstance(plain['A'], dict)
assert not isinstance(plain['A'], defaultdict)


Recursive conversion is especially useful before serialization or when returning data from an API boundary.


# Final review

When you see repetitive code such as:

```python
if key not in d:
    d[key] = ...
```

ask what the missing value represents.

If the missing value is a stable, repeatable part of the data model, a `defaultdict` may make the code clearer.


A practical decision table:

| Desired missing value | Factory |
|---|---|
| `0` | `int` |
| `0.0` | `float` |
| `[]` | `list` |
| `set()` | `set` |
| `{}` | `dict` |
| constant string/value | `lambda: value` |
| structured record | custom no-argument function |
| nested mapping | factory returning another `defaultdict` |


And remember the main caveat:

> `d[key]` on a missing key is both a lookup **and potentially a mutation**.

During construction, that is often exactly why `defaultdict` is useful.

During read-only reporting, `.get()` or a normal dictionary may be safer.
